# EyeCU 4.0 — Football Detector Training (Google Colab)

Trains and compares detectors on the EyeCU dataset built by `tools/build_dataset.py`.

**Classes:** `0 player`, `1 goalkeeper`, `2 referee`, `3 ball`. Team identity is decided
later by `trackers/team_assigner.py` — it is *not* a detector class.

**Runtime → Change runtime type → GPU** before running.

| # | Model | imgsz | Purpose |
|---|---|---|---|
| A | YOLO26s | 960 | **primary candidate** |
| B | YOLO26s | 1280 | ball / small-object comparison |
| C | YOLO26n | 960 | speed comparison |

Selection is on **ball / referee / goalkeeper recall + FPS**, not overall mAP.

## 1. Environment

In [ ]:
!pip install -q -U ultralytics
!nvidia-smi

import torch, ultralytics
from ultralytics import YOLO

print('ultralytics', ultralytics.__version__)
print('torch      ', torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime -> Change runtime type -> GPU, then rerun.')
print('gpu        ', torch.cuda.get_device_name(0))

## 2. Dataset

Upload `football_dataset.zip` (from `python tools/build_dataset.py --zip`) to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH  = '/content/drive/MyDrive/EyeCU/football_dataset.zip'  # <-- edit me
DRIVE_OUT = '/content/drive/MyDrive/EyeCU/runs'
DATA_ROOT = '/content/football_dataset'

In [ ]:
import os, shutil, yaml
from pathlib import Path

assert os.path.exists(ZIP_PATH), f'Not found: {ZIP_PATH}'
if os.path.exists(DATA_ROOT):
    shutil.rmtree(DATA_ROOT)
shutil.unpack_archive(ZIP_PATH, DATA_ROOT)

# build_dataset.py wrote an absolute local path; point it at the Colab copy.
YAML_PATH = f'{DATA_ROOT}/football.yaml'
cfg = yaml.safe_load(open(YAML_PATH))
cfg['path'] = DATA_ROOT
yaml.safe_dump(cfg, open(YAML_PATH, 'w'), sort_keys=False)

NAMES = [cfg['names'][i] for i in sorted(cfg['names'])]
print(cfg)

In [ ]:
# Sanity check: counts per split, no shared image, and no shared SOURCE MATCH.
from collections import Counter, defaultdict

stems, sources = {}, defaultdict(set)
for split in ('train', 'val', 'test'):
    img_dir = Path(DATA_ROOT) / 'images' / split
    if not img_dir.is_dir():
        continue                      # a split can be absent by design
    lbl_dir = Path(DATA_ROOT) / 'labels' / split
    imgs = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    stems[split] = {p.stem for p in imgs}
    counts, empty = Counter(), 0
    for p in imgs:
        # Filenames are <source_match>_<frame_index>, so the match is the
        # stem minus its trailing index.
        sources[split].add(p.stem.rsplit('_', 1)[0])
        txt = lbl_dir / f'{p.stem}.txt'
        lines = [l for l in txt.read_text().splitlines() if l.strip()] if txt.exists() else []
        if not lines:
            empty += 1
        counts.update(NAMES[int(float(l.split()[0]))] for l in lines)
    print(f'{split:<6}{len(imgs):>5} images, {len(sources[split])} matches, {empty} empty  ' +
          '  '.join(f'{n}={counts.get(n, 0)}' for n in NAMES))

# 1. no image in two splits
for a, b in [(a, b) for a in stems for b in stems if a < b]:
    overlap = stems[a] & stems[b]
    assert not overlap, f'IMAGE LEAK: {len(overlap)} images shared between {a} and {b}'

# 2. no source MATCH in two splits -- the one that actually matters. Frames
#    sampled seconds apart are near-identical, so a match spanning the
#    boundary inflates validation scores even when every filename is distinct.
for a, b in [(a, b) for a in sources for b in sources if a < b]:
    shared = sources[a] & sources[b]
    assert not shared, f'MATCH LEAK: {sorted(shared)} appear in both {a} and {b}'

print()
print('OK: no image and no source match appears in more than one split.')
for split in sources:
    print(f'  {split:<6} matches: {sorted(sources[split])}')

## 3. Train + evaluate

`evaluate()` records mAP50, mAP50-95, per-class precision/recall and inference FPS.
Run the 10-epoch pilot first to time a full run before committing to one.

In [ ]:
import json, time

RESULTS = {}


def evaluate(model, imgsz, split='val'):
    """mAP + per-class precision/recall + inference FPS."""
    m = model.val(data=YAML_PATH, imgsz=imgsz, split=split, verbose=False)

    per_class = {}
    for i, c in enumerate(m.box.ap_class_index):
        per_class[NAMES[int(c)]] = {
            'precision': float(m.box.p[i]),
            'recall':    float(m.box.r[i]),
            'mAP50':     float(m.box.ap50[i]),
            'mAP50_95':  float(m.box.ap[i]),
        }
    # A class absent from this split reports nothing. Keep it as None so it
    # cannot be misread as a score of zero.
    for name in NAMES:
        per_class.setdefault(name, None)

    speed = m.speed  # milliseconds per image
    per_img_ms = sum(speed.get(k, 0) for k in ('preprocess', 'inference', 'postprocess'))
    return {
        'split': split,
        'mAP50': float(m.box.map50),
        'mAP50_95': float(m.box.map),
        'precision_mean': float(m.box.mp),
        'recall_mean': float(m.box.mr),
        'per_class': per_class,
        'inference_ms': round(speed.get('inference', 0), 2),
        'total_ms_per_image': round(per_img_ms, 2),
        'fps': round(1000 / per_img_ms, 1) if per_img_ms else None,
    }


def train(name, weights, imgsz=960, epochs=80, patience=20, batch=-1, **kw):
    print(f'\n=== {name}: {weights} @ {imgsz}px, {epochs} epochs ===')
    try:
        model = YOLO(weights)          # pretrained, never from scratch
    except Exception as e:
        print(f'!! {weights} unavailable in ultralytics {ultralytics.__version__}: {e}')
        print('   Skipping. Substitute yolo11s.pt / yolo11n.pt if YOLO26 is not published yet.')
        return None

    t0 = time.time()
    model.train(data=YAML_PATH, epochs=epochs, imgsz=imgsz, batch=batch,
                patience=patience, project='/content/runs', name=name,
                exist_ok=True, seed=0, **kw)
    minutes = (time.time() - t0) / 60

    RESULTS[name] = {'weights': weights, 'imgsz': imgsz, 'epochs': epochs,
                     'train_minutes': round(minutes, 1),
                     'best_pt': f'/content/runs/{name}/weights/best.pt',
                     **evaluate(model, imgsz)}
    print(json.dumps(RESULTS[name], indent=2))
    return RESULTS[name]

In [ ]:
# PILOT — 10 epochs to confirm the dataset trains and to time a full run.
train('pilot', 'yolo26s.pt', imgsz=960, epochs=10, patience=0)

## 4. Experiments

Colab disconnects on idle — run one cell at a time and keep the tab open.

In [ ]:
# A — primary candidate
train('A_yolo26s_960', 'yolo26s.pt', imgsz=960, epochs=80)

In [ ]:
# B — ball / small-object comparison. If this OOMs, set a fixed small batch
# (e.g. batch=4). Only prefer 1280 if the ball-recall gain beats the FPS cost.
train('B_yolo26s_1280', 'yolo26s.pt', imgsz=1280, epochs=80)

In [ ]:
# C — speed comparison
train('C_yolo26n_960', 'yolo26n.pt', imgsz=960, epochs=80)

## 5. Compare

In [ ]:
import pandas as pd

summary = pd.DataFrame([{
    'run': k, 'imgsz': v['imgsz'],
    'mAP50': round(v['mAP50'], 4), 'mAP50-95': round(v['mAP50_95'], 4),
    'FPS': v['fps'], 'train_min': v['train_minutes'],
} for k, v in RESULTS.items()]).set_index('run')
display(summary.sort_values('mAP50-95', ascending=False))

# The metrics that actually decide selection.
rows = []
for run, v in RESULTS.items():
    row = {'run': run, 'FPS': v['fps']}
    for c in NAMES:
        pc = v['per_class'].get(c)
        row[f'{c}_R'] = round(pc['recall'], 3) if pc else None
        row[f'{c}_P'] = round(pc['precision'], 3) if pc else None
    rows.append(row)
print('\nPer-class precision/recall — ball, referee and goalkeeper recall decide this:')
display(pd.DataFrame(rows).set_index('run'))

In [ ]:
# Score the chosen model ONCE on the held-out test matches.
# Choose from the val tables above; using test to choose stops it being held out.
BEST = 'A_yolo26s_960'   # <-- set deliberately after reading the tables

best_model = YOLO(RESULTS[BEST]['best_pt'])
test_metrics = evaluate(best_model, RESULTS[BEST]['imgsz'], split='test')
print(json.dumps(test_metrics, indent=2))

## 6. Export and save to Drive

In [ ]:
onnx_path = best_model.export(format='onnx', imgsz=RESULTS[BEST]['imgsz'], simplify=True)

os.makedirs(DRIVE_OUT, exist_ok=True)
for src in (RESULTS[BEST]['best_pt'], str(onnx_path)):
    dst = os.path.join(DRIVE_OUT, f'{BEST}_' + os.path.basename(src))
    shutil.copy2(src, dst)
    print('saved', dst)

results_path = os.path.join(DRIVE_OUT, 'experiments.json')
json.dump({'selected': BEST, 'test': test_metrics, 'experiments': RESULTS},
          open(results_path, 'w'), indent=2)
print('saved', results_path)

## 7. Back in the repo

1. Download `best.pt` from Drive to the repo root, e.g. `eyecu_football.pt`.
2. Run the pipeline against it — no Roboflow, no network:

   ```bash
   python run_pipeline.py --input input-videos/short.mp4 \
       --yolo-model eyecu_football.pt --imgsz 960 --max-frames 300
   ```
3. Confirm `goalkeeper` and `referee` now appear in `tracks` — a COCO model
   cannot produce them at all.
4. `pytest` — the regression suite must still pass.

Then continue with docs/archive/TODO_legacy.md Phase 4 (post-processing) and Phase 5 (tracking).
The detector must be frozen before trackers are compared, otherwise you cannot
tell which change caused which result.

⚠️ **Speed/distance output is UNCALIBRATED.** `pixels_per_meter=12.0` in
`trackers/speed_distance.py` is an unvalidated guess, so km/h and metre figures
are not trustworthy however good the detector becomes. Detection quality and
speed calibration are separate problems.